In [ ]:
"""
Real Estate Demand Intelligence — central configuration.
"""
import os
from pathlib import Path
try:
    import yaml
except ImportError:
    yaml = None

try:
    dbutils.widgets.dropdown("ENV", "dev", ["dev", "test", "prod"], "Environment")
    dbutils.widgets.text("LAKEHOUSE", "", "Lakehouse name")
    dbutils.widgets.dropdown("MOCK_IPSTACK", "true", ["true", "false"], "Mock IPstack")
    dbutils.widgets.text("LOOKBACK_DAYS", "90", "Lookback days")
    dbutils.widgets.text("VIEWS_FILE", "property_views_500.csv", "Property views CSV")
    os.environ["ENV"] = dbutils.widgets.get("ENV").strip().lower()
    if dbutils.widgets.get("LAKEHOUSE").strip():
        os.environ["LAKEHOUSE"] = dbutils.widgets.get("LAKEHOUSE").strip()
    os.environ["MOCK_IPSTACK"] = dbutils.widgets.get("MOCK_IPSTACK").strip().lower()
    os.environ["LOOKBACK_DAYS"] = dbutils.widgets.get("LOOKBACK_DAYS").strip()
    os.environ["VIEWS_FILE"] = dbutils.widgets.get("VIEWS_FILE").strip()
except NameError:
    pass

def _load_env_yaml():
    if yaml is None:
        return {}
    for cfg_path in [Path("config/environments.yaml"), Path("/Workspace/Repos/real-estate/config/environments.yaml")]:
        if cfg_path.exists():
            with open(cfg_path) as f:
                return yaml.safe_load(f) or {}
    return {}

_ENV_CFG = _load_env_yaml()

class Config:
    def __init__(self, env=None, lakehouse=None, mock_ipstack=None, lookback_days=None, views_file=None):
        self.env = (env or os.getenv("ENV") or "dev").strip().lower()
        ec = _ENV_CFG.get(self.env, {})
        self.lakehouse = (lakehouse or os.getenv("LAKEHOUSE") or ec.get("lakehouse") or f"lh_realestate_{self.env}").strip()
        self.schema = ec.get("schema", "dbo")
        self.market = ec.get("market", "AU")
        self.mock_ipstack = str(mock_ipstack if mock_ipstack is not None else os.getenv("MOCK_IPSTACK", ec.get("mock_ipstack", True))).lower() == "true"
        self.lookback_days = int(lookback_days or os.getenv("LOOKBACK_DAYS") or ec.get("lookback_days", 90))
        self.views_file = (views_file or os.getenv("VIEWS_FILE") or ec.get("views_file") or "property_views_500.csv").strip()
        self.listings_file = ec.get("listings_file", "listings.csv")
        self.files_base = f"/lakehouse/{self.lakehouse}/Files"
        self.listings_path = f"{self.files_base}/raw/sample/{self.listings_file}"
        self.views_path = f"{self.files_base}/raw/sample/{self.views_file}"
        self.fixture_base = f"{self.files_base}/fixtures/ipstack"
        # Bronze
        self.bronze_listings = "bronze_listings"
        self.bronze_property_views = "bronze_property_views"
        self.bronze_ipstack_raw = "bronze_ipstack_raw"
        self.bronze_ipstack_errors = "bronze_ipstack_errors"
        # Silver
        self.silver_listings = "silver_listings"
        self.silver_ip_dim = "silver_ip_dim"
        self.silver_visits_enriched = "silver_visits_enriched"
        # Gold
        self.gold_suburb_interest = "gold_suburb_interest"
        self.gold_property_type_by_suburb = "gold_property_type_by_suburb"
        self.gold_price_engagement = "gold_price_engagement"
        self.gold_conversion_gaps = "gold_conversion_gaps"
        self.gold_property_trends = "gold_property_trends"
        self.gold_region_type_preference = "gold_region_type_preference"
        self.gold_repeat_interest = "gold_repeat_interest"
        self.gold_interstate_flow = "gold_interstate_flow"

    def table_fqn(self, table: str) -> str:
        return f"{self.lakehouse}.{self.schema}.{table}"

conf = Config()
from pyspark.sql import SparkSession
spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print(f"Config env={conf.env} lakehouse={conf.lakehouse} market={conf.market} mock={conf.mock_ipstack}")

